## 1. Import Library

In [2]:
!pip install pandas numpy scikit-learn tensorflow matplotlib seaborn tensorflowjs joblib --quiet

In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import tensorflowjs as tfjs

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

ModuleNotFoundError: No module named 'pandas'

## 2. Load Dataset (MovieLens 100K)

In [ ]:
# 1. Tentukan URL dataset MovieLens 100K
url_ratings = "http://files.grouplens.org/datasets/movielens/ml-100k/u.data"
url_movies  = "http://files.grouplens.org/datasets/movielens/ml-100k/u.item"

# 2. Load ratings
rating_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
df_ratings  = pd.read_csv(url_ratings, sep='\t', names=rating_cols)

# 3. Load movie titles (akan digunakan nanti untuk rekomendasi)
movie_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url']
movies_df  = pd.read_csv(url_movies, sep='|', names=movie_cols,
                          usecols=range(5), encoding='latin-1')

# 4. Gabungkan kedua dataset
df_ratings = pd.merge(df_ratings, movies_df, on='movie_id')

print(f"Total Interaksi (Ratings) : {len(df_ratings):,}")
print(f"Total User Unik           : {df_ratings['user_id'].nunique():,}")
print(f"Total Movie Unik          : {df_ratings['movie_id'].nunique():,}")
print(f"Rentang Rating            : {df_ratings['rating'].min()} – {df_ratings['rating'].max()}")
print()
print("Sampel Data:")
df_ratings.head()

In [ ]:
# Visualisasi distribusi rating
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribusi rating
rating_counts = df_ratings['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribusi Rating', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating (Bintang)')
axes[0].set_ylabel('Jumlah')
for i, v in zip(rating_counts.index, rating_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# Top 10 user paling aktif
top_users = df_ratings['user_id'].value_counts().head(10)
axes[1].barh(top_users.index.astype(str), top_users.values, color='coral')
axes[1].set_title('Top 10 User Paling Aktif', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Rating')
axes[1].set_ylabel('User ID')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
# 1. Encoding User ID dan Movie ID ke indeks integer berurutan
user_encoder  = LabelEncoder()
movie_encoder = LabelEncoder()

df_ratings['user_encoded']  = user_encoder.fit_transform(df_ratings['user_id'])
df_ratings['movie_encoded'] = movie_encoder.fit_transform(df_ratings['movie_id'])

num_users  = len(user_encoder.classes_)
num_movies = len(movie_encoder.classes_)

print(f"Jumlah User Unik  : {num_users}")
print(f"Jumlah Movie Unik : {num_movies}")

# 2. Normalisasi Rating ke skala 0–1
min_rating = df_ratings['rating'].min()
max_rating = df_ratings['rating'].max()
df_ratings['rating_norm'] = (df_ratings['rating'] - min_rating) / (max_rating - min_rating)

print(f"\nRating asli     : {min_rating} – {max_rating}")
print(f"Rating ternorm  : {df_ratings['rating_norm'].min():.1f} – {df_ratings['rating_norm'].max():.1f}")

# 3. Split Data (80% Train, 20% Validation)
X = df_ratings[['user_encoded', 'movie_encoded']].values
y = df_ratings['rating_norm'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nData Training   : {len(X_train):,} baris")
print(f"Data Validation : {len(X_val):,} baris")

## 4. Bangun Model Neural Collaborative Filtering (NCF)

In [ ]:
EMBEDDING_SIZE = 50

user_input     = Input(shape=(1,), name='User_Input')
user_embedding = Embedding(input_dim=num_users, output_dim=EMBEDDING_SIZE,
                           name='User_Embedding')(user_input)
user_vector    = Flatten(name='User_Flatten')(user_embedding)

movie_input     = Input(shape=(1,), name='Movie_Input')
movie_embedding = Embedding(input_dim=num_movies, output_dim=EMBEDDING_SIZE,
                            name='Movie_Embedding')(movie_input)
movie_vector    = Flatten(name='Movie_Flatten')(movie_embedding)

# PENGGABUNGAN 
concat = Concatenate(name='Concatenate')([user_vector, movie_vector])

# MULTI-LAYER PERCEPTRON
fc1      = Dense(128, activation='relu', name='FC_128')(concat)
drop1    = Dropout(0.2, name='Dropout_1')(fc1)

fc2      = Dense(64, activation='relu', name='FC_64')(drop1)
drop2    = Dropout(0.2, name='Dropout_2')(fc2)

fc3      = Dense(32, activation='relu', name='FC_32')(drop2)

# OUTPUT 
output   = Dense(1, activation='sigmoid', name='Output')(fc3)

# BUILD MODEL
model = Model(inputs=[user_input, movie_input], outputs=output, name='NCF_Model')
model.summary()

## 5. Training Model

In [ ]:
# Kompilasi
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

# Early Stopping — berhenti jika val_loss tidak membaik 3 epoch berturut-turut
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    x=[X_train[:, 0], X_train[:, 1]],
    y=y_train,
    batch_size=128,
    epochs=15,
    validation_data=([X_val[:, 0], X_val[:, 1]], y_val),
    callbacks=[early_stopping],
    verbose=1
)

## 6. Evaluasi Model

In [ ]:
# Grafik Learning Curve: Loss (MSE) + MAE 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Learning Curve — Proses Training Model NCF', fontsize=14, fontweight='bold')

# Grafik Loss (MSE) 
axes[0].plot(history.history['loss'],     label='Training Loss',   color='steelblue',  linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='orangered',  linewidth=2, linestyle='--')
axes[0].set_title('Loss (Mean Squared Error)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# --- Grafik MAE ---
axes[1].plot(history.history['mae'],     label='Training MAE',   color='seagreen',  linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='darkorange', linewidth=2, linestyle='--')
axes[1].set_title('MAE (Mean Absolute Error)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Hitung Metrik Evaluasi
y_pred_norm = model.predict([X_val[:, 0], X_val[:, 1]], verbose=0)

# Denormalisasi kembali ke skala 1–5
y_val_real  = (y_val             * (max_rating - min_rating)) + min_rating
y_pred_real = (y_pred_norm.flatten() * (max_rating - min_rating)) + min_rating

rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
mae  = mean_absolute_error(y_val_real, y_pred_real)

print("HASIL EVALUASI MODEL NCF")
print(f"  RMSE (Root Mean Squared Error) : {rmse:.4f}")
print(f"  MAE  (Mean Absolute Error)     : {mae:.4f}")
print(f"  → Rata-rata prediksi meleset sekitar {mae:.2f} bintang")
print(f"    dari rating asli user (skala 1–5)")

In [ ]:
#  Visualisasi: Prediksi vs Aktual + Distribusi Error
errors = y_pred_real - y_val_real

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analisis Prediksi Model', fontsize=14, fontweight='bold')

# Scatter: Prediksi vs Aktual
axes[0].scatter(y_val_real, y_pred_real, alpha=0.15, s=8, color='steelblue')
axes[0].plot([1, 5], [1, 5], 'r--', linewidth=2, label='Prediksi Sempurna')
axes[0].set_xlabel('Rating Asli', fontsize=11)
axes[0].set_ylabel('Rating Prediksi', fontsize=11)
axes[0].set_title('Prediksi vs Aktual', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Histogram Distribusi Error
axes[1].hist(errors, bins=40, color='salmon', edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Error = 0')
axes[1].axvline(errors.mean(), color='blue', linestyle='-', linewidth=1.5,
                label=f'Mean Error = {errors.mean():.3f}')
axes[1].set_xlabel('Error (Prediksi − Aktual)', fontsize=11)
axes[1].set_ylabel('Frekuensi', fontsize=11)
axes[1].set_title('Distribusi Error Prediksi', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Fungsi Rekomendasi Film

In [ ]:
def get_top_n_recommendations(user_id, n=10, verbose=True):
    """
    Menghasilkan rekomendasi Top-N film untuk user tertentu.

    Parameters
    ----------
    user_id : int
        ID user asli dari dataset MovieLens.
    n : int
        Jumlah rekomendasi yang diinginkan (default: 10).
    verbose : bool
        Tampilkan info proses jika True.

    Returns
    -------
    pd.DataFrame
        DataFrame berisi 'title', 'movie_id', 'predicted_rating'.
        None jika user_id tidak valid.
    """
    # 1. Validasi user_id
    if user_id not in user_encoder.classes_:
        print(f"User ID {user_id} tidak ditemukan dalam dataset!")
        print(f"   Coba salah satu dari: {list(user_encoder.classes_[:10])} ...")
        return None

    # 2. Encode user ID
    encoded_user = user_encoder.transform([user_id])[0]

    # 3. Cari film yang sudah & belum ditonton
    movies_watched     = set(df_ratings[df_ratings['user_id'] == user_id]['movie_id'].tolist())
    all_movies         = df_ratings['movie_id'].unique()
    movies_not_watched = [m for m in all_movies if m not in movies_watched]

    # 4. Siapkan input prediksi
    user_array  = np.array([encoded_user] * len(movies_not_watched))
    movie_array = movie_encoder.transform(movies_not_watched)

    # 5. Prediksi & denormalisasi
    pred_norm = model.predict([user_array, movie_array], verbose=0).flatten()
    pred_real = (pred_norm * (max_rating - min_rating)) + min_rating

    # 6. Rangking & ambil Top-N
    recs_df = pd.DataFrame({'movie_id': movies_not_watched, 'predicted_rating': pred_real})
    top_n   = recs_df.sort_values('predicted_rating', ascending=False).head(n)
    result  = pd.merge(top_n, movies_df[['movie_id', 'title']], on='movie_id')
    result  = result[['title', 'movie_id', 'predicted_rating']].reset_index(drop=True)
    result.index += 1  # Mulai dari 1

    return result

In [ ]:
recommendations = get_top_n_recommendations(user_id=42, n=10)

In [ ]:
# Coba user ID yang tidak ada untuk lihat validasi error
get_top_n_recommendations(user_id=9999, n=5)

# Coba user lain yang valid
recommendations_user100 = get_top_n_recommendations(user_id=100, n=5)
if recommendations_user100 is not None:
    print(f"\n🎬 Top 5 untuk User 100:")
    print(recommendations_user100.to_string())

## 8. Simpan Model & Encoder

In [ ]:
import os
import json


OUTPUT_DIR = 'ncf_model_output'
TFJS_DIR   = os.path.join(OUTPUT_DIR, 'tfjs_model')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TFJS_DIR, exist_ok=True)

# 1. Simpan model LANGSUNG ke format TensorFlow.js
# Ini akan menghasilkan file model.json dan file .bin (bobot model) di dalam folder TFJS_DIR
tfjs.converters.save_keras_model(model, TFJS_DIR)
print(f"Model berhasil disimpan ke format TFJS : {TFJS_DIR}")

# 2. Simpan metadata (format JSON sangat cocok untuk dibaca oleh JavaScript di frontend)
# (Kode joblib .pkl dihapus karena tidak bisa dibaca oleh TFJS/JavaScript)
metadata = {
    "min_rating"   : float(min_rating),
    "max_rating"   : float(max_rating),
    "num_users"    : int(num_users),
    "num_movies"   : int(num_movies),
    "embedding_size": EMBEDDING_SIZE,
    "user_id_map"  : {int(enc): int(orig) for enc, orig in enumerate(user_encoder.classes_)},
    "movie_id_map" : {int(enc): int(orig) for enc, orig in enumerate(movie_encoder.classes_)}
}

meta_path = os.path.join(OUTPUT_DIR, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
    
print(f"Metadata JSON disimpan                 : {meta_path}")

In [ ]:
def export_keras_to_tfjs(model, output_dir):
    """
    Mengekspor model Keras ke format TensorFlow.js LayersModel.
    
    Output:
      - model.json        : Arsitektur model + manifest weights
      - weights.bin       : Binary weights (float32)
    
    Cara load di Node.js:
      const tf = require('@tensorflow/tfjs-node');
      const model = await tf.loadLayersModel('file://path/to/model.json');
    """
    os.makedirs(output_dir, exist_ok=True)
    
    weight_specs   = []
    weight_buffers = []
    
    for layer in model.layers:
        layer_weights = layer.get_weights()
        keras_weights = layer.weights
        
        for idx, (w_array, w_var) in enumerate(zip(layer_weights, keras_weights)):
            w_f32 = w_array.astype(np.float32)
            
            # Nama weight sesuai format TFJS
            w_name = w_var.name
            if ':' in w_name:
                w_name = w_name.split(':')[0]  # Hapus ":0" suffix
            
            weight_specs.append({
                "name" : w_name,
                "shape": list(w_f32.shape),
                "dtype": "float32"
            })
            weight_buffers.append(w_f32.flatten().tobytes())
    
    # 2. Gabungkan semua weights menjadi satu buffer binary
    combined_buffer = b''.join(weight_buffers)
    
    # 3. Simpan file .bin
    bin_filename = 'group1-shard1of1.bin'
    bin_path     = os.path.join(output_dir, bin_filename)
    with open(bin_path, 'wb') as f:
        f.write(combined_buffer)
    
    # 4. Build weights manifest
    weights_manifest = [{
        "paths"  : [bin_filename],
        "weights": weight_specs
    }]
    
    # 5. Ambil arsitektur model dalam format Keras JSON
    model_config = json.loads(model.to_json())
    
    # 6. Susun model.json sesuai spesifikasi TFJS
    model_json = {
        "format"          : "layers-model",
        "generatedBy"     : f"keras v{tf.keras.__version__}",
        "convertedBy"     : "Custom TFJS Exporter (NCF Project)",
        "modelTopology"   : model_config,
        "weightsManifest" : weights_manifest
    }
    
    # 7. Simpan model.json
    json_path = os.path.join(output_dir, 'model.json')
    with open(json_path, 'w') as f:
        json.dump(model_json, f, indent=2)
    
    # 8. Ringkasan output
    bin_size_kb = len(combined_buffer) / 1024
    print(f"  📄 model.json      → {json_path}")
    print(f"  🗜️  weights .bin   → {bin_path} ({bin_size_kb:.1f} KB)")
    print(f"  📦 Total weights   → {len(weight_specs)} tensor")
    
    return json_path, bin_path


print()
json_path, bin_path = export_keras_to_tfjs(model, TFJS_DIR)

print()
print(f"   Folder: {TFJS_DIR}/")

In [ ]:
print("\nSTRUKTUR FILE OUTPUT")
print("=" * 50)
for root, dirs, files in os.walk(OUTPUT_DIR):
    level   = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent  = '  ' * level
    print(f"{indent} {os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        size  = os.path.getsize(fpath)
        size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.2f} MB"
        print(f"{sub_indent}📄 {file}  ({size_str})")

## 9. Cara Menggunakan Model di Node.js

Setelah export selesai, gunakan file dari folder `tfjs_model/`.

In [ ]:
nodejs_guide = """
╔══════════════════════════════════════════════════════════════════════╗
║          PANDUAN PENGGUNAAN MODEL DI NODE.JS                        ║
╚══════════════════════════════════════════════════════════════════════╝

📦 STEP 1 — Install dependency
──────────────────────────────
  npm install @tensorflow/tfjs-node

📂 STEP 2 — Struktur folder di proyek Node.js
─────────────────────────────────────────────
  my-node-app/
  ├── tfjs_model/
  │   ├── model.json
  │   └── group1-shard1of1.bin
  ├── metadata.json
  └── recommend.js

📝 STEP 3 — Kode recommend.js
──────────────────────────────
  const tf  = require('@tensorflow/tfjs-node');
  const fs  = require('fs');
  const path = require('path');

  // Load model dan metadata
  async function loadModel() {
    const model    = await tf.loadLayersModel('file://./tfjs_model/model.json');
    const metadata = JSON.parse(fs.readFileSync('./metadata.json', 'utf8'));
    return { model, metadata };
  }

  // Fungsi prediksi rating untuk satu pasang user-movie
  async function predictRating(model, metadata, userId, movieId) {
    // Cari encoded index
    const userMap  = metadata.user_id_map;   // { encodedIdx: originalId }
    const movieMap = metadata.movie_id_map;

    // Balik map: originalId -> encodedIdx
    const userEnc  = Object.entries(userMap).find(([,v]) => v === userId)?.[0];
    const movieEnc = Object.entries(movieMap).find(([,v]) => v === movieId)?.[0];

    if (userEnc === undefined || movieEnc === undefined) {
      throw new Error('User ID atau Movie ID tidak ditemukan dalam mapping');
    }

    // Buat tensor input
    const userTensor  = tf.tensor2d([[parseInt(userEnc)]],  [1, 1], 'int32');
    const movieTensor = tf.tensor2d([[parseInt(movieEnc)]], [1, 1], 'int32');

    // Prediksi
    const predNorm = model.predict([userTensor, movieTensor]);
    const predArr  = await predNorm.data();

    // Denormalisasi ke skala 1–5
    const { min_rating, max_rating } = metadata;
    const ratingReal = predArr[0] * (max_rating - min_rating) + min_rating;

    // Bersihkan tensor dari memori
    userTensor.dispose();
    movieTensor.dispose();
    predNorm.dispose();

    return ratingReal;
  }

  // Jalankan
  (async () => {
    const { model, metadata } = await loadModel();
    console.log('✅ Model loaded successfully');
    console.log(`   Users  : ${metadata.num_users}`);
    console.log(`   Movies : ${metadata.num_movies}`);

    // Contoh prediksi: rating user 42 untuk movie 1
    const rating = await predictRating(model, metadata, 42, 1);
    console.log(`\n⭐ Predicted rating for User 42 → Movie 1 : ${rating.toFixed(2)}`);
  })();

▶️  STEP 4 — Jalankan
─────────────────────
  node recommend.js
"""
print(nodejs_guide)

## 10. Load Ulang & Verifikasi Model

In [ ]:
# Verifikasi model bisa di-load ulang dan menghasilkan prediksi yang konsisten
from tensorflow.keras.models import load_model

print("🔄 Memuat ulang model dari file .keras ...")
model_loaded = load_model(keras_path)

# Bandingkan prediksi model asli vs model yang di-load ulang
sample_users  = X_val[:5, 0]
sample_movies = X_val[:5, 1]

pred_original = model.predict([sample_users, sample_movies], verbose=0).flatten()
pred_reloaded = model_loaded.predict([sample_users, sample_movies], verbose=0).flatten()

print("\n✅ Perbandingan prediksi (5 sampel pertama):")
print(f"{'Sample':>8} | {'Original':>10} | {'Reloaded':>10} | {'Selisih':>10}")
print("-" * 48)
for i, (orig, relo) in enumerate(zip(pred_original, pred_reloaded)):
    diff = abs(orig - relo)
    print(f"{i+1:>8} | {orig:>10.6f} | {relo:>10.6f} | {diff:>10.2e}")

max_diff = np.max(np.abs(pred_original - pred_reloaded))
print(f"\n✅ Max perbedaan prediksi: {max_diff:.2e}  (mendekati nol = konsisten)")